# 📊 SÍNTESE VISUAL 2 — Latência e Memória por Modelo

In [ ]:
import json, sys, numpy as np
sys.path.insert(0, '..')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

METRICS_DIR = Path('experiments_results/metrics')
FIGURES_DIR = Path('experiments_results/figures')

plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'text.color':'#f0f6fc','axes.labelcolor':'#f0f6fc',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'axes.edgecolor':'#30363d','grid.color':'#30363d','grid.alpha':0.5,
})

def load(nb_id):
    p = METRICS_DIR/f'{nb_id}_results.json'
    return json.load(open(p)) if p.exists() else {}

def best_pauc(data):
    for key in ['avg_prot_a','pauc01']:
        if key in data: return float(data[key])
    if 'protocol_a' in data: return float(data['protocol_a'].get('pauc01',0))
    if 'spec' in data: return float(data['spec'].get('avg_prot_a',0))
    if 'models' in data:
        vals = [v.get('pauc01',0) for v in data['models'].values() if isinstance(v,dict)]
        if vals: return float(max(vals))
    if 'model' in data:
        m = data['model']
        if isinstance(m,dict): return float(m.get('pauc01',0))
    if 'ranking' in data:
        vals = [v.get('pauc01',0) for v in data['ranking'].values()]
        if vals: return float(max(vals))
    return 0.0

all_data = {f'nb{i:02d}': load(f'nb{i:02d}') for i in range(1,29)}
print(f"✅ {sum(1 for v in all_data.values() if v)} resultados carregados")


In [ ]:
# Coleta latência e memória de modelos-chave
nb01=load('nb01'); nb03=load('nb03'); nb12=load('nb12')
nb22=load('nb22'); nb27=load('nb27')

model_data = {}
for key, nb_d in [('nb01',nb01),('nb03',nb03)]:
    for k, v in nb_d.get('models',{}).items():
        if isinstance(v,dict) and v.get('latency_ms') and v.get('memory_mb'):
            model_data[v.get('label',k)] = {'lat': v['latency_ms'], 'mem': v['memory_mb'],
                                              'pauc': v.get('pauc01',0)}

# Adiciona protocolos finais
for prot, key in [('XGBoost (Prot.A)','protocol_a'),('GMM (Prot.B)','protocol_b')]:
    met = nb22.get(key, {})
    if met.get('latency_ms') and met.get('memory_mb'):
        model_data[prot] = {'lat': met['latency_ms'], 'mem': met['memory_mb'],
                             'pauc': met.get('pauc01',0)}

if model_data:
    labels = list(model_data.keys())
    lats = [model_data[l]['lat'] for l in labels]
    mems = [model_data[l]['mem'] for l in labels]
    paucs = [model_data[l]['pauc'] for l in labels]

    fig, ax = plt.subplots(figsize=(9,6))
    scatter_colors = ['#2ea043' if (l<=50 and m<=4) else '#da3633'
                      for l,m in zip(lats,mems)]
    sc = ax.scatter(lats, mems, c=scatter_colors, s=120, zorder=5, alpha=0.9)
    ax.axvline(50, color='#f0883e', ls='--', lw=1.5, alpha=0.8, label='Lat≤50ms')
    ax.axhline(4,  color='#f0883e', ls=':',  lw=1.5, alpha=0.8, label='Mem≤4MB')
    for lbl, x, y, p in zip(labels, lats, mems, paucs):
        ax.annotate(f'{lbl}\npAUC={p:.2f}', (x,y), fontsize=7.5,
                    xytext=(6,6), textcoords='offset points', color='white')
    ax.set_xlabel('Latência (ms)'); ax.set_ylabel('Memória (MB)')
    ax.set_title('Eficiência: Latência × Memória por Modelo', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
    plt.tight_layout()
    plt.savefig('experiments_results/figures/visual2_latency_memory.png', dpi=130,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
else:
    print("Dados de latência/memória insuficientes — verifique se os experimentos rodaram.")
